# Transformation

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW standard_concept_mapping AS
WITH ranked AS (
  SELECT
    concept.vocabulary_id         AS source_vocabulary_id,   -- ICD9CM / ICD10CM / SNOMED
    concept.concept_code          AS source_concept_code,
    concept.concept_id            AS source_concept_id,
    standard_concept.concept_id   AS standard_concept_id,
    standard_concept.concept_name AS standard_concept_name,
    concept_relationship.valid_start_date AS rel_valid_start_date,
    ROW_NUMBER() OVER (
      PARTITION BY concept.vocabulary_id, concept.concept_code
      ORDER BY concept_relationship.valid_start_date DESC, standard_concept.concept_id ASC
    ) AS rn
  FROM _exponent.omop.concept
  JOIN _exponent.omop.concept_relationship
    ON concept_relationship.concept_id_1 = concept.concept_id
   AND concept_relationship.relationship_id = 'Maps to'
   AND concept_relationship.invalid_reason IS NULL
  JOIN _exponent.omop.concept AS standard_concept
    ON standard_concept.concept_id = concept_relationship.concept_id_2
   AND standard_concept.standard_concept = 'S'
   AND standard_concept.domain_id = 'Condition'
   AND standard_concept.invalid_reason IS NULL
  WHERE concept.vocabulary_id IN ('ICD9CM', 'ICD10CM', 'SNOMED')
    AND concept.invalid_reason IS NULL
)
SELECT
  source_vocabulary_id,
  source_concept_code,
  source_concept_id,
  standard_concept_id,
  standard_concept_name
FROM ranked
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW problem_diagnosis AS
WITH problems AS (
SELECT 
CAST(id AS BIGINT) AS id,
entryname,
NULLIF(TRIM(REGEXP_REPLACE(Snomed3CODE,'[\\s\\u00A0]+', '')), '') AS SnomedCode,
NULLIF(TRIM(REGEXP_REPLACE(ICD10DiagnosisCode,'[\\s\\u00A0]+', '')), '') AS ICD10DiagnosisCode,
NULLIF(TRIM(REGEXP_REPLACE(ICD9DiagnosisCODE,'[\\s\\u00A0]+', '')), '') AS ICD9DiagnosisCODE
FROM
_exponent._bronze_allscripts_tw_works_vw.dbo_problem_de)
,problem_diagnosis AS (
  SELECT 
  id,
  entryname,
  COALESCE(SnomedCode, ICD10DiagnosisCode, ICD9DiagnosisCODE) AS DiagnosisCode,
    CASE 
    WHEN SnomedCode IS NOT NULL THEN 'SNOMED' 
    WHEN ICD10DiagnosisCode IS NOT NULL THEN 'ICD10CM' 
    WHEN ICD9DiagnosisCODE IS NOT NULL THEN 'ICD9CM' 
  END AS DiagnosisCodeType
FROM problems
)
SELECT 
problem_diagnosis.*,
standard_concept_mapping.*
FROM problem_diagnosis
LEFT JOIN standard_concept_mapping
  ON standard_concept_mapping.source_vocabulary_id = problem_diagnosis.DiagnosisCodeType
 AND standard_concept_mapping.source_concept_code = problem_diagnosis.DiagnosisCode


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_condition_occurrence AS
SELECT
  CONCAT_WS(
       CHR(31),
       'allscripts_tw',
       'dbo_encounter_diagnosis',
       'id',
       CAST(dbo_encounter_diagnosis.id AS BIGINT)
  ) AS condition_occurrence_source_value, -- staging key
  source_to_person.person_id,
  COALESCE(
    standard_concept_mapping.standard_concept_id,
    problem_diagnosis.standard_concept_id,
    0
  ) AS condition_concept_id,
  CAST(dbo_encounter.dttm AS DATE)      AS condition_start_date,
  CAST(dbo_encounter.dttm AS TIMESTAMP) AS condition_start_datetime,
  NULL AS condition_end_date,
  NULL AS condition_end_datetime,
  CASE
    WHEN dbo_encounter_diagnosis.DiagnosisType IN ('ICD9','ICD10') THEN 32020
    WHEN dbo_encounter_diagnosis.DiagnosisType = 'PROBLEM' THEN 38000245
    ELSE 32817
  END AS condition_type_concept_id,
  NULL AS condition_status_concept_id,
  NULL AS stop_reason,
  NULL AS provider_id,
  NULL AS visit_occurrence_id,
  NULL AS visit_detail_id,
  -- visit staging key: only ICD9/ICD10 rows have an encounter-linked visit
  -- PROBLEM rows have no direct encounter link, so visit_occurrence_source_value is NULL for them
  CASE
    WHEN dbo_encounter_diagnosis.DiagnosisType IN ('ICD9', 'ICD10')
    THEN CONCAT_WS(
           CHR(31),
           'allscripts_tw',
           'dbo_visit',
           'id',
           CAST(dbo_encounter.visitid AS BIGINT)
         )
    ELSE NULL
  END AS visit_occurrence_source_value,
  COALESCE(
    NULLIF(TRIM(REGEXP_REPLACE(dbo_ICD9_Diagnosis_DE.EntryCode,  '[\\s\\u00A0]+', '')), ''),
    NULLIF(TRIM(REGEXP_REPLACE(dbo_ICD10_Diagnosis_DE.EntryCode, '[\\s\\u00A0]+', '')), ''),
    problem_diagnosis.DiagnosisCode
  ) AS condition_source_value,
  COALESCE(
    standard_concept_mapping.source_concept_id,
    problem_diagnosis.source_concept_id,
    0
  ) AS condition_source_concept_id,

  NULL AS condition_status_source_value,
  'allscripts_tw' AS source_system

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_encounter

LEFT JOIN _exponent._bronze_allscripts_tw_works.dbo_encounter_diagnosis
  ON dbo_encounter_diagnosis.EncounterID = dbo_encounter.id

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_ICD10_Diagnosis_DE
  ON dbo_ICD10_Diagnosis_DE.id = dbo_encounter_diagnosis.DiagnosisDE
 AND dbo_encounter_diagnosis.DiagnosisType = 'ICD10'

LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_ICD9_Diagnosis_DE
  ON dbo_ICD9_Diagnosis_DE.id = dbo_encounter_diagnosis.DiagnosisDE
 AND dbo_encounter_diagnosis.DiagnosisType = 'ICD9'

LEFT JOIN problem_diagnosis
  ON problem_diagnosis.id = dbo_encounter_diagnosis.DiagnosisDE
 AND dbo_encounter_diagnosis.DiagnosisType = 'PROBLEM'

LEFT JOIN standard_concept_mapping
  ON dbo_encounter_diagnosis.DiagnosisType IN ('ICD9','ICD10')
 AND standard_concept_mapping.source_vocabulary_id = CASE
      WHEN dbo_encounter_diagnosis.DiagnosisType = 'ICD9'  THEN 'ICD9CM'
      WHEN dbo_encounter_diagnosis.DiagnosisType = 'ICD10' THEN 'ICD10CM'
    END
 AND standard_concept_mapping.source_concept_code = COALESCE(
      NULLIF(TRIM(REGEXP_REPLACE(dbo_ICD9_Diagnosis_DE.EntryCode,  '[\\s\\u00A0]+', '')), ''),
      NULLIF(TRIM(REGEXP_REPLACE(dbo_ICD10_Diagnosis_DE.EntryCode, '[\\s\\u00A0]+', '')), '')
    )

LEFT JOIN _exponent._bronze_allscripts_tw_works.dbo_person
  ON dbo_person.id = dbo_encounter.patientid

LEFT JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value = CONCAT_WS(
       chr(31),
       'allscripts_tw',
       'dbo_person',
       'id',
       CAST(dbo_person.id AS BIGINT)
     )
 AND source_to_person.active_flag = TRUE

WHERE 1=1
AND dbo_encounter_diagnosis.id IS NOT NULL
AND source_to_person.person_id IS NOT NULL;

# %sql
# ALTER TABLE _exponent.omop_silver.condition_occurrence
# ADD COLUMN visit_occurrence_source_value STRING;

In [0]:
%sql
MERGE INTO _exponent.omop_silver.condition_occurrence AS target
USING silver_condition_occurrence AS source
ON target.condition_occurrence_source_value = source.condition_occurrence_source_value

WHEN MATCHED AND NOT (
     target.person_id                        <=> source.person_id
 AND target.condition_concept_id             <=> source.condition_concept_id
 AND target.condition_start_date             <=> source.condition_start_date
 AND target.condition_start_datetime         <=> source.condition_start_datetime
 AND target.condition_end_date               <=> source.condition_end_date
 AND target.condition_end_datetime           <=> source.condition_end_datetime
 AND target.condition_type_concept_id        <=> source.condition_type_concept_id
 AND target.condition_status_concept_id      <=> source.condition_status_concept_id
 AND target.stop_reason                      <=> source.stop_reason
 AND target.provider_id                      <=> source.provider_id
 AND target.visit_occurrence_id              <=> source.visit_occurrence_id
 AND target.visit_detail_id                  <=> source.visit_detail_id
 AND target.visit_occurrence_source_value    <=> source.visit_occurrence_source_value
 AND target.condition_source_value           <=> source.condition_source_value
 AND target.condition_source_concept_id      <=> source.condition_source_concept_id
 AND target.condition_status_source_value    <=> source.condition_status_source_value
 AND target.source_system                    <=> source.source_system
) THEN UPDATE SET
  target.condition_occurrence_source_value = source.condition_occurrence_source_value,
  target.person_id                        = source.person_id,
  target.condition_concept_id             = source.condition_concept_id,
  target.condition_start_date             = source.condition_start_date,
  target.condition_start_datetime         = source.condition_start_datetime,
  target.condition_end_date               = source.condition_end_date,
  target.condition_end_datetime           = source.condition_end_datetime,
  target.condition_type_concept_id        = source.condition_type_concept_id,
  target.condition_status_concept_id      = source.condition_status_concept_id,
  target.stop_reason                      = source.stop_reason,
  target.provider_id                      = source.provider_id,
  target.visit_occurrence_id              = source.visit_occurrence_id,
  target.visit_detail_id                  = source.visit_detail_id,
  target.visit_occurrence_source_value    = source.visit_occurrence_source_value,
  target.condition_source_value           = source.condition_source_value,
  target.condition_source_concept_id      = source.condition_source_concept_id,
  target.condition_status_source_value    = source.condition_status_source_value,
  target.source_system                    = source.source_system,
  target.last_mod_tsp                     = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_source_value,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  visit_occurrence_source_value,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value,
  source_system,
  last_mod_tsp
) VALUES (
  source.condition_occurrence_source_value,
  source.person_id,
  source.condition_concept_id,
  source.condition_start_date,
  source.condition_start_datetime,
  source.condition_end_date,
  source.condition_end_datetime,
  source.condition_type_concept_id,
  source.condition_status_concept_id,
  source.stop_reason,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.visit_occurrence_source_value,
  source.condition_source_value,
  source.condition_source_concept_id,
  source.condition_status_source_value,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_condition_occurrence (
    source_system,
    condition_occurrence_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    silver_condition.source_system,
    silver_condition.condition_occurrence_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(silver_condition.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        condition_occurrence_source_value,
        last_mod_tsp
    FROM _exponent.omop_silver.condition_occurrence
    WHERE condition_occurrence_source_value IS NOT NULL
) AS silver_condition
LEFT ANTI JOIN _exponent.omop_mapping.source_to_condition_occurrence AS existing_condition
  ON silver_condition.condition_occurrence_source_value = existing_condition.condition_occurrence_source_value;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW gold AS
SELECT
  source_to_condition_occurrence.condition_occurrence_id,
  condition_occurrence.person_id,
  condition_occurrence.condition_concept_id,
  condition_occurrence.condition_start_date,
  condition_occurrence.condition_start_datetime,
  condition_occurrence.condition_end_date,
  condition_occurrence.condition_end_datetime,
  condition_occurrence.condition_type_concept_id,
  condition_occurrence.condition_status_concept_id,
  condition_occurrence.stop_reason,
  condition_occurrence.provider_id,
  stvo.visit_occurrence_id,
  condition_occurrence.visit_detail_id,
  condition_occurrence.condition_source_value,
  condition_occurrence.condition_source_concept_id,
  condition_occurrence.condition_status_source_value

FROM _exponent.omop_silver.condition_occurrence

JOIN _exponent.omop_mapping.source_to_condition_occurrence
  ON condition_occurrence.condition_occurrence_source_value =
     source_to_condition_occurrence.condition_occurrence_source_value
 AND source_to_condition_occurrence.active_flag = TRUE

LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
  ON condition_occurrence.visit_occurrence_source_value = stvo.visit_occurrence_source_value
 AND stvo.source_system = 'allscripts_tw'
 AND stvo.active_flag   = TRUE

WHERE condition_occurrence.source_system = 'allscripts_tw';

In [0]:
%sql
-- MERGE INTO _exponent.omop.condition_occurrence AS target
MERGE INTO _exponent.omop_tw.condition_occurrence AS target
USING gold AS source
ON target.condition_occurrence_id = source.condition_occurrence_id

WHEN MATCHED AND NOT (
     target.person_id                     <=> source.person_id
 AND target.condition_concept_id          <=> source.condition_concept_id
 AND target.condition_start_date          <=> source.condition_start_date
 AND target.condition_start_datetime      <=> source.condition_start_datetime
 AND target.condition_end_date            <=> source.condition_end_date
 AND target.condition_end_datetime        <=> source.condition_end_datetime
 AND target.condition_type_concept_id     <=> source.condition_type_concept_id
 AND target.condition_status_concept_id   <=> source.condition_status_concept_id
 AND target.stop_reason                   <=> source.stop_reason
 AND target.provider_id                   <=> source.provider_id
 AND target.visit_occurrence_id           <=> source.visit_occurrence_id
 AND target.visit_detail_id               <=> source.visit_detail_id
 AND target.condition_source_value        <=> source.condition_source_value
 AND target.condition_source_concept_id   <=> source.condition_source_concept_id
 AND target.condition_status_source_value <=> source.condition_status_source_value
) THEN UPDATE SET
  target.person_id                     = source.person_id,
  target.condition_concept_id          = source.condition_concept_id,
  target.condition_start_date          = source.condition_start_date,
  target.condition_start_datetime      = source.condition_start_datetime,
  target.condition_end_date            = source.condition_end_date,
  target.condition_end_datetime        = source.condition_end_datetime,
  target.condition_type_concept_id     = source.condition_type_concept_id,
  target.condition_status_concept_id   = source.condition_status_concept_id,
  target.stop_reason                   = source.stop_reason,
  target.provider_id                   = source.provider_id,
  target.visit_occurrence_id           = source.visit_occurrence_id,
  target.visit_detail_id               = source.visit_detail_id,
  target.condition_source_value        = source.condition_source_value,
  target.condition_source_concept_id   = source.condition_source_concept_id,
  target.condition_status_source_value = source.condition_status_source_value

WHEN NOT MATCHED THEN INSERT (
  condition_occurrence_id,
  person_id,
  condition_concept_id,
  condition_start_date,
  condition_start_datetime,
  condition_end_date,
  condition_end_datetime,
  condition_type_concept_id,
  condition_status_concept_id,
  stop_reason,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  condition_source_value,
  condition_source_concept_id,
  condition_status_source_value
) VALUES (
  source.condition_occurrence_id,
  source.person_id,
  source.condition_concept_id,
  source.condition_start_date,
  source.condition_start_datetime,
  source.condition_end_date,
  source.condition_end_datetime,
  source.condition_type_concept_id,
  source.condition_status_concept_id,
  source.stop_reason,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.condition_source_value,
  source.condition_source_concept_id,
  source.condition_status_source_value
);

SELECT
  COUNT(*)                    AS total_rows,
  COUNT(visit_occurrence_id)  AS has_visit_id
FROM _exponent.omop_tw.condition_occurrence;

SELECT *
FROM _exponent.omop_tw.condition_occurrence
limit 100;

SELECT *
FROM _exponent.omop_tw.condition_occurrence
WHERE condition_type_concept_id = 32020
LIMIT 20;

SELECT 
  co.condition_occurrence_source_value,
  co.visit_occurrence_source_value,
  stvo.visit_occurrence_source_value AS mapping_key,
  stvo.visit_occurrence_id
FROM _exponent.omop_silver.condition_occurrence co
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
  ON co.visit_occurrence_source_value = stvo.visit_occurrence_source_value
 AND stvo.source_system = 'allscripts_tw'
 AND stvo.active_flag = TRUE
WHERE co.source_system = 'allscripts_tw'
  AND co.condition_type_concept_id = 32020
LIMIT 10;